In [1]:
!pip install spotipy python-dotenv python-docx

In [2]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyOAuth

env_path = r"C:\Users\naiar\OneDrive\Documents\GitHub\time-capsule\.env.txt"
print("File exists:", os.path.exists(env_path))

load_dotenv(env_path)
print("CLIENT_ID:", os.environ.get("SPOTIPY_CLIENT_ID"))

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    scope="playlist-modify-private playlist-modify-public playlist-read-private"
))

me = sp.current_user()
print(f"Logged in as: {me['display_name']}")

File exists: True
CLIENT_ID: 02dabb9aced748f89676a98ba10e8369
Logged in as: naiara.soares


In [3]:
def search_track(sp, song, artist):
    """
    Searches Spotify for a track matching the song and artist.
    Returns the track's URI, or None if nothing was found.
    """
    query = f"track:{song} artist:{artist}"
    results = sp.search(q=query, type="track", limit=1)
    items = results["tracks"]["items"]

    if not items:
        return None

    track = items[0]
    print(f"Found: {track['name']} - {track['artists'][0]['name']}")
    return track["uri"]

In [4]:
uri = search_track(sp, "Yellow", "Coldplay")
print(uri)

Found: Yellow - Coldplay
spotify:track:6O2cts0GC32etUCSWyp51r


In [5]:
def add_to_time_capsule(sp, song, artist):
    """
    Searches for the song and adds it to the Time Capsule playlist.
    Returns True if it was added successfully, False otherwise.
    """
    track_uri = search_track(sp, song, artist)

    if track_uri is None:
        print("Song not found on Spotify.")
        return False

    playlist_id = find_or_create_playlist(sp)
    sp.playlist_add_items(playlist_id, [track_uri])
    return True

In [6]:
def find_or_create_playlist(sp, playlist_name="Time Capsule"):
    """
    Looks through the user's playlists for one called "Time Capsule".
    If it doesn't exist yet, creates it.
    Returns the playlist ID.
    """
    playlists = sp.current_user_playlists()["items"]

    for playlist in playlists:
        if playlist["name"] == playlist_name:
            return playlist["id"]

    new_playlist = sp.current_user_playlist_create(
        name=playlist_name,
        public=False,
        description="A musical time capsule - one memory at a time."
    )
    return new_playlist["id"]

In [8]:
try:
    playlist_id = find_or_create_playlist(sp)
    print("Playlist ID:", playlist_id)
except Exception as e:
    print("ERROR:", e)

Playlist ID: 6XYY2p2IRfo5R2FYXsdSnt


In [9]:
import os
from datetime import date
from docx import Document

def create_memory_document(song, artist, memory_text):
    """
    Creates a Word document for a single memory entry, with today's
    date in the filename(YYYY-MM-DD).
    Returns the file path of the document that was created.
    """
    today_str = date.today().isoformat() #e.g. "2026-08-12"
    doc = Document()
    doc.add_heading("Time Capsule", level=0)
    doc.add_paragraph(f"Date sealed: {today_str}")
    doc.add_paragraph("Song: {song}")
    doc.add_paragraph(f"Artist: {artist}")
    doc.add_paragraph("")
    doc.add_heading("Time Memory", level=1)
    doc.add_paragraph(memory_text)

    filename = f"TimeCapsule_{today_str}.docx"
    doc.save(filename)


    return filename


In [10]:
memory_text = "This song was playing when I got my acceptance letter college."
filepath = create_memory_document(song, artist, memory_text)
print(f"Saved: {filepath}")

Saved: TimeCapsule_2026-08-12.docx


In [11]:
song = input("Viajando Por El Mundo")
artist = input("Karol G, Manu Chao")
memory_text = input("This was my first solo song time trip by my own")

filepath = create_memory_document(song, artist, memory_text)
print(f"Saved: {filepath}")

Viajando Por El Mundo 
Karol G, Manu Chao 
This was my first solo song time trip by my own 


Saved: TimeCapsule_2026-08-12.docx


In [12]:
import shutil

def move_to_logs(filepath, logs_folder="logs"):
    """
    Moves a file into the logs folder, creating the folder if it doesn't exist yet. 
    Returns the new path of the file.
    """

    os.makedirs(logs_folder, exist_ok=True)
    filename = os.path.basename(filepath)
    new_path = os.path.join(logs_folder, filename)
    shutil.move(filepath, new_path)
    return new_path    
    

In [13]:
new_path = move_to_logs(filepath)
print(f"Moved to: {new_path}")

Moved to: logs\TimeCapsule_2026-08-12.docx


In [28]:
def add_memory():
    """
    Full flow: ask the user for a memory, save it as a Word document,
    try to add the song to the Time Capsule playlist, and only move the document into the logs folder if 
    that upload succeeds.
    """
    song = input("Song title: ")
    artist = input("Artist: ")
    memory_text = input("Your memory: ")

    filepath = create_memory_document(song, artist, memory_text)
    print(f"Memory saved as: {filepath}")

    success = add_to_time_capsule(sp, song, artist)

    if success:
        new_path = move_to_logs(filepath)
        print(f"Song added to Spotify! Memory archived at: {new_path}")
    else:
        print(f"Song not added to Spotify. Memory stays at: {filepath} (not archived)")

In [29]:
add_memory()


Song title:  Verano Lento
Artist:  Carlos Sadness, Jósean Log
Your memory:  Last holidays in Madrid


Memory saved as: TimeCapsule_2026-08-12.docx
Found: Verano Lento - Carlos Sadness
Song added to Spotify! Memory archived at: logs\TimeCapsule_2026-08-12.docx
